In [1]:
"""
================================================================================
NLP EXAM PRACTICE — COMPREHENSIVE REFERENCE SCRIPT
================================================================================
Allowed libraries ONLY:
    nltk, pandas, numpy, sklearn, hmmlearn, pycrfsuite, sklearn_crfsuite, PyPDF2
(collections / math / re / io are Python standard library — always allowed)

Install (if needed):
    pip install nltk pandas numpy scikit-learn hmmlearn python-crfsuite \
                sklearn-crfsuite PyPDF2

Sections (search for "SECTION" to jump around):
    1.  Reading data from files (txt / csv / pdf)
    2.  Phases of NLP
    3.  Preprocessing (lowercase, stopword removal, tokenization, etc.)
    4.  Word embeddings (co-occurrence matrix + SVD, from scratch)
    5.  N-grams + Perplexity + Sentence probability
    6.  TF-IDF + LSI (using SVD)
    7.  POS tagging using HMM (supervised counts + Viterbi, and hmmlearn)
    8.  POS tagging using CRF (pycrfsuite / sklearn_crfsuite / from scratch)
    9.  Byte-Pair Encoding (BPE) — from scratch
    10. Edit Distance (Levenshtein) — from scratch

Each section is self-contained: copy-paste whichever block you need in the exam.
Run this whole file with `python nlp_exam_practice.py` to see every demo execute.
================================================================================
"""

import math
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.util import ngrams as nltk_ngrams

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

# ------------------------------------------------------------------------------
# One-time NLTK downloads (safe to call every run; no-ops if already present)
# ------------------------------------------------------------------------------
for pkg in ["punkt", "punkt_tab", "stopwords", "averaged_perceptron_tagger",
            "averaged_perceptron_tagger_eng", "treebank", "wordnet",
            "omw-1.4"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass


# ==============================================================================
# SECTION 1: READING DATA FROM FILES
# ==============================================================================

def read_txt_file(path):
    """Read a plain text file and return its content as a single string."""
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def read_txt_lines(path):
    """Read a plain text file and return a list of lines (no trailing \\n)."""
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f.readlines()]


def read_csv_file(path, text_column=None):
    """
    Read a CSV file using pandas.
    If text_column is given, also return that column as a list of strings.
    """
    df = pd.read_csv(path)
    texts = df[text_column].astype(str).tolist() if text_column else None
    return df, texts


def read_pdf_file(path):
    """
    Read a PDF file using PyPDF2 and return the full extracted text.
    """
    from PyPDF2 import PdfReader
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted + "\n"
    return text


def section1_demo():
    print("\n=== SECTION 1: Reading data from files ===")
    print("(Demonstration only — shows function signatures.")
    print(" In the exam, call read_txt_file('data.txt'), read_csv_file('data.csv', 'text'),")
    print(" or read_pdf_file('data.pdf') on your actual files.)")
    print("read_txt_file, read_txt_lines, read_csv_file, read_pdf_file are defined above.")


# ==============================================================================
# SECTION 2: PHASES OF NLP
# ==============================================================================
# 1. Lexical / Morphological Analysis  -> tokenization, identifying word structure
# 2. Syntactic Analysis (Parsing)      -> grammar, POS tags, parse trees
# 3. Semantic Analysis                 -> literal meaning of words/sentences
# 4. Discourse Integration             -> meaning of a sentence depends on context
#                                          of preceding/following sentences
# 5. Pragmatic Analysis                -> real-world/contextual/intended meaning

NLP_PHASES = {
    "Lexical/Morphological Analysis": "Breaks text into tokens/words and studies "
                                       "word structure (stems, prefixes, suffixes).",
    "Syntactic Analysis": "Analyzes grammatical structure of sentences (POS tags, "
                           "parse trees, dependency relations).",
    "Semantic Analysis": "Extracts literal/dictionary meaning of words and how "
                          "they combine into sentence meaning.",
    "Discourse Integration": "Interprets a sentence's meaning using the context "
                              "of surrounding sentences.",
    "Pragmatic Analysis": "Interprets intended meaning using real-world "
                           "knowledge, tone, and context (beyond literal text).",
}


def section2_demo():
    print("\n=== SECTION 2: Phases of NLP ===")
    text = "Apple is looking at buying a UK startup for $1 billion."

    for phase, desc in NLP_PHASES.items():
        print(f"\n-- {phase} --\n{desc}")

    # Lexical: tokenization
    tokens = word_tokenize(text)
    print("\n[Lexical] Tokens:", tokens)

    # Syntactic: POS tagging
    pos_tags = nltk.pos_tag(tokens)
    print("[Syntactic] POS tags:", pos_tags)

    # Semantic: WordNet definitions (word sense)
    from nltk.corpus import wordnet as wn
    synsets = wn.synsets("bank")
    if synsets:
        print("[Semantic] 'bank' first sense definition:", synsets[0].definition())

    # Discourse / Pragmatic are conceptual — shown via comments, not single-call code
    print("[Discourse] e.g. 'He scored a goal. It was his third.' — 'It' resolved via discourse.")
    print("[Pragmatic] e.g. 'Can you pass the salt?' is a request, not a literal question.")


# ==============================================================================
# SECTION 3: PREPROCESSING
# ==============================================================================

def to_lowercase(text):
    """Lowercase all characters in the text."""
    return text.lower()


def remove_punctuation(text):
    """Remove punctuation using regex."""
    return re.sub(r"[^\w\s]", "", text)


def remove_numbers(text):
    """Remove standalone digits."""
    return re.sub(r"\d+", "", text)


def tokenize(text):
    """Word tokenize using nltk."""
    return word_tokenize(text)


def remove_stopwords(tokens, language="english"):
    """Remove stopwords from a list of tokens using NLTK's stopword list."""
    stop_words = set(stopwords.words(language))
    return [tok for tok in tokens if tok.lower() not in stop_words]


def stem_tokens(tokens):
    """Porter stemming."""
    from nltk.stem import PorterStemmer
    ps = PorterStemmer()
    return [ps.stem(tok) for tok in tokens]


def lemmatize_tokens(tokens):
    """WordNet lemmatization."""
    from nltk.stem import WordNetLemmatizer
    lemmatizer = WordNetLemmatizer()
    return [lemmatizer.lemmatize(tok) for tok in tokens]


def preprocess_pipeline(text, do_stem=False, do_lemmatize=False):
    """
    Full standard preprocessing pipeline:
    lowercase -> remove punctuation -> tokenize -> remove stopwords -> (stem/lemmatize)
    """
    text = to_lowercase(text)
    text = remove_punctuation(text)
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    if do_stem:
        tokens = stem_tokens(tokens)
    if do_lemmatize:
        tokens = lemmatize_tokens(tokens)
    return tokens


def section3_demo():
    print("\n=== SECTION 3: Preprocessing ===")
    text = "The Quick, Brown Foxes are RUNNING and jumping over 2 lazy dogs!"
    print("Original:", text)
    print("Lowercase:", to_lowercase(text))
    print("No punctuation:", remove_punctuation(to_lowercase(text)))
    tokens = tokenize(remove_punctuation(to_lowercase(text)))
    print("Tokens:", tokens)
    print("Stopwords removed:", remove_stopwords(tokens))
    print("Stemmed:", stem_tokens(remove_stopwords(tokens)))
    print("Lemmatized:", lemmatize_tokens(remove_stopwords(tokens)))
    print("Full pipeline:", preprocess_pipeline(text, do_lemmatize=True))


# ==============================================================================
# SECTION 4: WORD EMBEDDINGS (co-occurrence matrix + SVD — built with sklearn)
# ==============================================================================
# Since gensim/word2vec libraries aren't in the allowed list, we build a classic
# "distributional" embedding: a word x context co-occurrence matrix reduced with
# TruncatedSVD (this is the core idea behind LSA-style / early word-embeddings).

def build_cooccurrence_matrix(sentences, window_size=2):
    """
    Build a word x word co-occurrence matrix from a list of tokenized sentences.
    sentences: list[list[str]] (already tokenized, lowercase)
    Returns: (matrix as np.ndarray, vocab list, word2idx dict)
    """
    vocab = sorted({w for sent in sentences for w in sent})
    word2idx = {w: i for i, w in enumerate(vocab)}
    V = len(vocab)
    co_matrix = np.zeros((V, V), dtype=np.float64)

    for sent in sentences:
        for i, word in enumerate(sent):
            wi = word2idx[word]
            start = max(0, i - window_size)
            end = min(len(sent), i + window_size + 1)
            for j in range(start, end):
                if j != i:
                    wj = word2idx[sent[j]]
                    co_matrix[wi, wj] += 1
    return co_matrix, vocab, word2idx


def get_word_embeddings(co_matrix, n_components=10):
    """Reduce the co-occurrence matrix to dense embeddings using TruncatedSVD."""
    n_components = min(n_components, co_matrix.shape[1] - 1) if co_matrix.shape[1] > 1 else 1
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    embeddings = svd.fit_transform(co_matrix)
    return embeddings, svd


def most_similar_word(word, embeddings, vocab, word2idx, top_n=3):
    """Return top_n most similar words to `word` using cosine similarity."""
    if word not in word2idx:
        return []
    idx = word2idx[word]
    sims = cosine_similarity(embeddings[idx].reshape(1, -1), embeddings)[0]
    ranked = sorted(
        [(vocab[i], sims[i]) for i in range(len(vocab)) if i != idx],
        key=lambda x: -x[1],
    )
    return ranked[:top_n]


def section4_demo():
    print("\n=== SECTION 4: Word Embeddings (co-occurrence + SVD) ===")
    corpus = [
        "the king is a strong man",
        "the queen is a strong woman",
        "the man and the woman are royal",
        "the king and queen rule the kingdom",
    ]
    sentences = [preprocess_pipeline(s) for s in corpus]
    co_matrix, vocab, word2idx = build_cooccurrence_matrix(sentences, window_size=2)
    embeddings, svd = get_word_embeddings(co_matrix, n_components=5)

    print("Vocab:", vocab)
    print("Embedding shape:", embeddings.shape)
    if "king" in word2idx:
        print("Most similar to 'king':", most_similar_word("king", embeddings, vocab, word2idx))


# ==============================================================================
# SECTION 5: N-GRAMS + PERPLEXITY + SENTENCE PROBABILITY
# ==============================================================================

def build_ngram_model(tokenized_sentences, n=2):
    """
    Build an n-gram frequency model with sentence boundary tokens.
    Returns: (ngram_counts, context_counts, vocab_size)
    """
    ngram_counts = Counter()
    context_counts = Counter()
    vocab = set()

    for sent in tokenized_sentences:
        padded = ["<s>"] * (n - 1) + sent + ["</s>"]
        vocab.update(padded)
        for gram in nltk_ngrams(padded, n):
            ngram_counts[gram] += 1
            context_counts[gram[:-1]] += 1

    return ngram_counts, context_counts, len(vocab)


def ngram_prob(gram, ngram_counts, context_counts, vocab_size, smoothing=1.0):
    """Laplace (add-k) smoothed conditional probability P(w_n | w_1..w_n-1)."""
    context = gram[:-1]
    num = ngram_counts.get(gram, 0) + smoothing
    denom = context_counts.get(context, 0) + smoothing * vocab_size
    return num / denom


def sentence_probability(sentence_tokens, ngram_counts, context_counts, vocab_size, n=2):
    """Compute P(sentence) = product of n-gram conditional probabilities."""
    padded = ["<s>"] * (n - 1) + sentence_tokens + ["</s>"]
    prob = 1.0
    for gram in nltk_ngrams(padded, n):
        prob *= ngram_prob(gram, ngram_counts, context_counts, vocab_size)
    return prob


def sentence_log_probability(sentence_tokens, ngram_counts, context_counts, vocab_size, n=2):
    """Compute log P(sentence) — more numerically stable than raw probability."""
    padded = ["<s>"] * (n - 1) + sentence_tokens + ["</s>"]
    log_prob = 0.0
    for gram in nltk_ngrams(padded, n):
        p = ngram_prob(gram, ngram_counts, context_counts, vocab_size)
        log_prob += math.log(p)
    return log_prob


def perplexity(test_sentences, ngram_counts, context_counts, vocab_size, n=2):
    """
    Perplexity of a test set under the n-gram model:
        PP = exp( - (1/N) * sum(log P(w_i | context)) )
    where N = total number of n-grams scored across the whole test set.
    """
    total_log_prob = 0.0
    total_ngrams = 0
    for sent in test_sentences:
        padded = ["<s>"] * (n - 1) + sent + ["</s>"]
        for gram in nltk_ngrams(padded, n):
            p = ngram_prob(gram, ngram_counts, context_counts, vocab_size)
            total_log_prob += math.log(p)
            total_ngrams += 1
    avg_log_prob = total_log_prob / total_ngrams
    return math.exp(-avg_log_prob)


def section5_demo():
    print("\n=== SECTION 5: N-grams, Perplexity, Sentence Probability ===")
    train_corpus = [
        "i love natural language processing",
        "i love machine learning",
        "natural language processing is fun",
        "machine learning is powerful",
    ]
    train_tokens = [word_tokenize(s) for s in train_corpus]

    n = 2  # bigram model
    ngram_counts, context_counts, vocab_size = build_ngram_model(train_tokens, n=n)

    test_sentence = word_tokenize("i love natural language processing")
    prob = sentence_probability(test_sentence, ngram_counts, context_counts, vocab_size, n=n)
    log_prob = sentence_log_probability(test_sentence, ngram_counts, context_counts, vocab_size, n=n)
    print(f"P(sentence) = {prob:.10e}")
    print(f"log P(sentence) = {log_prob:.4f}")

    test_corpus = [word_tokenize("i love machine learning"),
                   word_tokenize("natural language processing is powerful")]
    pp = perplexity(test_corpus, ngram_counts, context_counts, vocab_size, n=n)
    print(f"Perplexity on test set = {pp:.4f}")


# ==============================================================================
# SECTION 6: TF-IDF + LSI (Latent Semantic Indexing) USING SVD
# ==============================================================================

def build_tfidf(corpus):
    """Build a TF-IDF matrix (docs x terms) using sklearn's TfidfVectorizer."""
    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf_matrix = vectorizer.fit_transform(corpus)
    return tfidf_matrix, vectorizer


def build_lsi(tfidf_matrix, n_components=2):
    """
    Latent Semantic Indexing: apply Truncated SVD to the TF-IDF matrix.
    tfidf_matrix (docs x terms) = U * Sigma * V^T
      - doc_topic_matrix  (U*Sigma) -> documents represented in topic space
      - term_topic_matrix (svd.components_) -> terms represented in topic space
    """
    n_components = min(n_components, min(tfidf_matrix.shape) - 1)
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    doc_topic_matrix = svd.fit_transform(tfidf_matrix)
    term_topic_matrix = svd.components_  # shape: (n_components, n_terms)
    return doc_topic_matrix, term_topic_matrix, svd


def top_terms_per_topic(vectorizer, term_topic_matrix, top_n=5):
    """Print the top contributing terms for each LSI topic/component."""
    terms = np.array(vectorizer.get_feature_names_out())
    topics = {}
    for i, comp in enumerate(term_topic_matrix):
        top_idx = np.argsort(comp)[::-1][:top_n]
        topics[f"Topic {i}"] = terms[top_idx].tolist()
    return topics


def section6_demo():
    print("\n=== SECTION 6: TF-IDF + LSI (SVD) ===")
    corpus = [
        "The cat sat on the mat",
        "The dog barked at the cat",
        "Dogs and cats are common pets",
        "The stock market rallied today",
        "Investors bought stocks and bonds",
    ]
    tfidf_matrix, vectorizer = build_tfidf(corpus)
    print("TF-IDF matrix shape (docs x terms):", tfidf_matrix.shape)

    doc_topic, term_topic, svd = build_lsi(tfidf_matrix, n_components=2)
    print("Document-topic matrix:\n", np.round(doc_topic, 3))
    print("Explained variance ratio:", svd.explained_variance_ratio_)
    print("Top terms per topic:", top_terms_per_topic(vectorizer, term_topic))


# ==============================================================================
# SECTION 7: POS TAGGING USING HMM
# ==============================================================================
# Approach A: Fully manual — supervised counts + hand-written Viterbi (most exam-safe)
# Approach B: Using hmmlearn.CategoricalHMM with manually-set parameters + decode()

def train_hmm_from_tagged_sents(tagged_sentences):
    """
    Estimate HMM parameters (transition, emission, initial) from a
    list of tagged sentences: [[(word, tag), ...], ...]
    Uses add-1 (Laplace) smoothing.
    """
    tags = sorted({tag for sent in tagged_sentences for _, tag in sent})
    words = sorted({word.lower() for sent in tagged_sentences for word, _ in sent})
    tag2idx = {t: i for i, t in enumerate(tags)}
    word2idx = {w: i for i, w in enumerate(words)}
    T, W = len(tags), len(words)

    initial_counts = np.ones(T)          # add-1 smoothing
    transition_counts = np.ones((T, T))
    emission_counts = np.ones((T, W))

    for sent in tagged_sentences:
        first_word, first_tag = sent[0]
        initial_counts[tag2idx[first_tag]] += 1
        for i, (word, tag) in enumerate(sent):
            emission_counts[tag2idx[tag], word2idx[word.lower()]] += 1
            if i > 0:
                prev_tag = sent[i - 1][1]
                transition_counts[tag2idx[prev_tag], tag2idx[tag]] += 1

    initial_probs = initial_counts / initial_counts.sum()
    transition_probs = transition_counts / transition_counts.sum(axis=1, keepdims=True)
    emission_probs = emission_counts / emission_counts.sum(axis=1, keepdims=True)

    return {
        "tags": tags, "words": words,
        "tag2idx": tag2idx, "word2idx": word2idx,
        "initial": initial_probs, "transition": transition_probs, "emission": emission_probs,
    }


def viterbi_decode(sentence_tokens, model):
    """Hand-written Viterbi algorithm for HMM POS tagging."""
    tags = model["tags"]
    T = len(tags)
    N = len(sentence_tokens)
    word2idx = model["word2idx"]

    def emit_prob(tag_idx, word):
        if word.lower() in word2idx:
            return model["emission"][tag_idx, word2idx[word.lower()]]
        return 1e-6  # unseen word fallback probability

    viterbi = np.zeros((T, N))
    backpointer = np.zeros((T, N), dtype=int)

    for t in range(T):
        viterbi[t, 0] = model["initial"][t] * emit_prob(t, sentence_tokens[0])

    for i in range(1, N):
        for t in range(T):
            probs = viterbi[:, i - 1] * model["transition"][:, t] * emit_prob(t, sentence_tokens[i])
            backpointer[t, i] = np.argmax(probs)
            viterbi[t, i] = np.max(probs)

    best_path = [int(np.argmax(viterbi[:, N - 1]))]
    for i in range(N - 1, 0, -1):
        best_path.insert(0, backpointer[best_path[0], i])

    return [tags[i] for i in best_path]


def hmm_pos_tag_hmmlearn(tagged_sentences, sentence_tokens):
    """
    Alternative: use hmmlearn.hmm.CategoricalHMM.
    hmmlearn's .fit() is unsupervised (EM), so for SUPERVISED POS tagging we
    instead estimate parameters ourselves (like above) and load them directly
    into the hmmlearn model, then call .decode() (Viterbi) for tagging.
    """
    from hmmlearn import hmm

    model_params = train_hmm_from_tagged_sents(tagged_sentences)
    tags, word2idx = model_params["tags"], model_params["word2idx"]
    T = len(tags)

    hmm_model = hmm.CategoricalHMM(n_components=T, n_iter=100, init_params="")
    hmm_model.startprob_ = model_params["initial"]
    hmm_model.transmat_ = model_params["transition"]
    hmm_model.emissionprob_ = model_params["emission"]
    hmm_model.n_features = len(model_params["words"])

    # Encode observation sequence as integer indices (unseen words -> index 0 fallback)
    obs_seq = np.array([[word2idx.get(w.lower(), 0)] for w in sentence_tokens])

    log_prob, state_seq = hmm_model.decode(obs_seq, algorithm="viterbi")
    predicted_tags = [tags[s] for s in state_seq]
    return predicted_tags, log_prob


def section7_demo():
    print("\n=== SECTION 7: POS Tagging using HMM ===")
    # Small toy tagged corpus (word, tag) — Penn-Treebank-style tags
    tagged_sentences = [
        [("the", "DT"), ("dog", "NN"), ("barks", "VBZ")],
        [("the", "DT"), ("cat", "NN"), ("sleeps", "VBZ")],
        [("a", "DT"), ("dog", "NN"), ("runs", "VBZ")],
        [("the", "DT"), ("cat", "NN"), ("runs", "VBZ")],
    ]
    model = train_hmm_from_tagged_sents(tagged_sentences)

    test_sentence = ["the", "dog", "sleeps"]
    tags_manual = viterbi_decode(test_sentence, model)
    print("Manual Viterbi tags:", list(zip(test_sentence, tags_manual)))

    try:
        tags_hmmlearn, log_prob = hmm_pos_tag_hmmlearn(tagged_sentences, test_sentence)
        print("hmmlearn Viterbi tags:", list(zip(test_sentence, tags_hmmlearn)))
        print("hmmlearn log-likelihood:", log_prob)
    except Exception as e:
        print("hmmlearn demo skipped (library issue):", e)


# ==============================================================================
# SECTION 8: POS TAGGING USING CRF (3 approaches)
# ==============================================================================

def word2features(sent, i):
    """
    Extract a feature dict for word at position i in a sentence.
    sent: list of words (strings)
    Common features: word identity, suffixes, casing, neighboring words.
    """
    word = sent[i]
    features = {
        "bias": 1.0,
        "word.lower()": word.lower(),
        "word[-3:]": word[-3:],
        "word[-2:]": word[-2:],
        "word.isupper()": word.isupper(),
        "word.istitle()": word.istitle(),
        "word.isdigit()": word.isdigit(),
    }
    if i > 0:
        prev_word = sent[i - 1]
        features.update({
            "-1:word.lower()": prev_word.lower(),
            "-1:word.istitle()": prev_word.istitle(),
        })
    else:
        features["BOS"] = True  # beginning of sentence

    if i < len(sent) - 1:
        next_word = sent[i + 1]
        features.update({
            "+1:word.lower()": next_word.lower(),
            "+1:word.istitle()": next_word.istitle(),
        })
    else:
        features["EOS"] = True  # end of sentence

    return features


def sent2features(sent):
    """sent: list of words -> list of feature dicts"""
    return [word2features(sent, i) for i in range(len(sent))]


def sent2labels(tagged_sent):
    """tagged_sent: list of (word, tag) -> list of tags"""
    return [tag for _, tag in tagged_sent]


def sent2tokens(tagged_sent):
    return [word for word, _ in tagged_sent]


# -- 8a. pycrfsuite ------------------------------------------------------------

def train_crf_pycrfsuite(tagged_sentences, model_path="pos_model.crfsuite"):
    import pycrfsuite

    X_train = [sent2features(sent2tokens(s)) for s in tagged_sentences]
    y_train = [sent2labels(s) for s in tagged_sentences]

    trainer = pycrfsuite.Trainer(verbose=False)
    for xseq, yseq in zip(X_train, y_train):
        trainer.append(xseq, yseq)

    trainer.set_params({
        "c1": 1.0,          # L1 penalty
        "c2": 1e-3,         # L2 penalty
        "max_iterations": 100,
        "feature.possible_transitions": True,
    })
    trainer.train(model_path)
    return model_path


def tag_with_pycrfsuite(model_path, sentence_tokens):
    import pycrfsuite

    tagger = pycrfsuite.Tagger()
    tagger.open(model_path)
    features = sent2features(sentence_tokens)
    predicted = tagger.tag(features)
    tagger.close()
    return predicted


# -- 8b. sklearn_crfsuite -------------------------------------------------------

def train_crf_sklearn(tagged_sentences):
    import sklearn_crfsuite

    X_train = [sent2features(sent2tokens(s)) for s in tagged_sentences]
    y_train = [sent2labels(s) for s in tagged_sentences]

    crf = sklearn_crfsuite.CRF(
        algorithm="lbfgs",
        c1=1.0,
        c2=1e-3,
        max_iterations=100,
        all_possible_transitions=True,
    )
    crf.fit(X_train, y_train)
    return crf


def tag_with_sklearn_crf(crf, sentence_tokens):
    features = [sent2features(sentence_tokens)]
    predicted = crf.predict(features)
    return predicted[0]


# -- 8c. CRF built from scratch (simplified linear-chain CRF via structured
#        perceptron — a standard exam-friendly approximation of full CRF
#        training, since true gradient-based CRF training with forward-backward
#        is lengthy; the perceptron uses the same feature+Viterbi machinery) ---

class SimpleCRF:
    """
    A simplified linear-chain sequence labeler trained with the
    structured perceptron algorithm (a discriminative, feature-based
    model in the same family as CRFs, minus the log-linear/softmax
    normalization and gradient-based training).
    """

    def __init__(self, feature_fn=word2features):
        self.feature_fn = feature_fn
        self.weights = defaultdict(float)   # (feature, tag) -> weight
        self.trans_weights = defaultdict(float)  # (prev_tag, tag) -> weight
        self.tags = set()

    def _score(self, feats, tag):
        return sum(self.weights[(f, tag)] * (v if isinstance(v, (int, float)) else 1.0)
                    for f, v in feats.items() if v not in (False,))

    def _viterbi(self, sent_tokens):
        feats_seq = [self.feature_fn(sent_tokens, i) for i in range(len(sent_tokens))]
        tags = list(self.tags)
        N = len(sent_tokens)
        T = len(tags)

        dp = np.full((T, N), -1e18)
        bp = np.zeros((T, N), dtype=int)

        for t in range(T):
            dp[t, 0] = self._score(feats_seq[0], tags[t])

        for i in range(1, N):
            for t in range(T):
                best_score, best_prev = -1e18, 0
                emit = self._score(feats_seq[i], tags[t])
                for p in range(T):
                    trans = self.trans_weights[(tags[p], tags[t])]
                    score = dp[p, i - 1] + trans + emit
                    if score > best_score:
                        best_score, best_prev = score, p
                dp[t, i] = best_score
                bp[t, i] = best_prev

        best_last = int(np.argmax(dp[:, N - 1]))
        path = [best_last]
        for i in range(N - 1, 0, -1):
            path.insert(0, bp[path[0], i])
        return [tags[i] for i in path]

    def fit(self, tagged_sentences, epochs=5):
        self.tags = {tag for sent in tagged_sentences for _, tag in sent}
        for epoch in range(epochs):
            for sent in tagged_sentences:
                tokens = sent2tokens(sent)
                gold_tags = sent2labels(sent)
                pred_tags = self._viterbi(tokens)
                if pred_tags != gold_tags:
                    feats_seq = [self.feature_fn(tokens, i) for i in range(len(tokens))]
                    for i in range(len(tokens)):
                        for f, v in feats_seq[i].items():
                            if v in (False,):
                                continue
                            self.weights[(f, gold_tags[i])] += 1.0
                            self.weights[(f, pred_tags[i])] -= 1.0
                        if i > 0:
                            self.trans_weights[(gold_tags[i - 1], gold_tags[i])] += 1.0
                            self.trans_weights[(pred_tags[i - 1], pred_tags[i])] -= 1.0
        return self

    def predict(self, sentence_tokens):
        return self._viterbi(sentence_tokens)


def section8_demo():
    print("\n=== SECTION 8: POS Tagging using CRF ===")
    tagged_sentences = [
        [("The", "DT"), ("dog", "NN"), ("barks", "VBZ")],
        [("The", "DT"), ("cat", "NN"), ("sleeps", "VBZ")],
        [("A", "DT"), ("dog", "NN"), ("runs", "VBZ")],
        [("The", "DT"), ("cat", "NN"), ("runs", "VBZ")],
        [("A", "DT"), ("cat", "NN"), ("barks", "VBZ")],
    ]
    test_tokens = ["The", "dog", "runs"]

    # 8a. pycrfsuite
    try:
        model_path = train_crf_pycrfsuite(tagged_sentences)
        preds = tag_with_pycrfsuite(model_path, test_tokens)
        print("pycrfsuite predicted tags:", list(zip(test_tokens, preds)))
    except Exception as e:
        print("pycrfsuite demo skipped:", e)

    # 8b. sklearn_crfsuite
    try:
        crf = train_crf_sklearn(tagged_sentences)
        preds = tag_with_sklearn_crf(crf, test_tokens)
        print("sklearn_crfsuite predicted tags:", list(zip(test_tokens, preds)))
    except Exception as e:
        print("sklearn_crfsuite demo skipped:", e)

    # 8c. From scratch (structured perceptron)
    simple_crf = SimpleCRF().fit(tagged_sentences, epochs=10)
    preds = simple_crf.predict(test_tokens)
    print("From-scratch CRF (perceptron) predicted tags:", list(zip(test_tokens, preds)))


# ==============================================================================
# SECTION 9: BYTE-PAIR ENCODING (BPE) — built from scratch
# ==============================================================================

def get_word_freqs(corpus_words):
    """
    corpus_words: list[str] raw words (already tokenized).
    Returns a dict mapping "char sequence with </w> end marker" -> frequency.
    e.g. "low" -> "l o w </w>" : 5
    """
    freqs = Counter(corpus_words)
    return {" ".join(list(word)) + " </w>": freq for word, freq in freqs.items()}


def get_pair_stats(vocab):
    """Count frequency of adjacent symbol pairs across the vocab."""
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs


def merge_vocab(pair, vocab):
    """Merge all occurrences of `pair` into a single symbol across the vocab."""
    bigram = " ".join(pair)
    replacement = "".join(pair)
    new_vocab = {}
    for word, freq in vocab.items():
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = freq
    return new_vocab


def byte_pair_encoding(corpus_words, num_merges=10):
    """
    Learn BPE merges from a list of words.
    Returns: (final_vocab, list_of_merges_in_order)
    """
    vocab = get_word_freqs(corpus_words)
    merges = []

    for _ in range(num_merges):
        pairs = get_pair_stats(vocab)
        if not pairs:
            break
        best_pair = max(pairs, key=pairs.get)
        vocab = merge_vocab(best_pair, vocab)
        merges.append(best_pair)

    return vocab, merges


def apply_bpe_to_word(word, merges):
    """Apply learned merges (in order) to segment a new word."""
    symbols = list(word) + ["</w>"]
    for pair in merges:
        i = 0
        new_symbols = []
        while i < len(symbols):
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
                new_symbols.append(symbols[i] + symbols[i + 1])
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols
    return symbols


def section9_demo():
    print("\n=== SECTION 9: Byte-Pair Encoding ===")
    corpus_words = ["low", "low", "lower", "lowest", "newer", "newer", "newest", "wide", "wider"]
    final_vocab, merges = byte_pair_encoding(corpus_words, num_merges=10)

    print("Learned merges (in order):")
    for i, m in enumerate(merges, 1):
        print(f"  {i}. {m}")

    print("Final subword vocab:")
    for word, freq in final_vocab.items():
        print(f"  {word!r}: {freq}")

    new_word = "lowering"
    segmented = apply_bpe_to_word(new_word, merges)
    print(f"BPE segmentation of unseen word '{new_word}':", segmented)


# ==============================================================================
# SECTION 10: EDIT DISTANCE (Levenshtein) — built from scratch
# ==============================================================================

def edit_distance(s1, s2, insert_cost=1, delete_cost=1, substitute_cost=1):
    """
    Classic dynamic-programming Levenshtein edit distance between s1 and s2.
    Returns (distance, dp_table) where dp_table can be used to trace the
    optimal sequence of operations.
    """
    m, n = len(s1), len(s2)
    dp = np.zeros((m + 1, n + 1), dtype=int)

    for i in range(m + 1):
        dp[i][0] = i * delete_cost
    for j in range(n + 1):
        dp[0][j] = j * insert_cost

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = min(
                    dp[i - 1][j] + delete_cost,      # deletion
                    dp[i][j - 1] + insert_cost,       # insertion
                    dp[i - 1][j - 1] + substitute_cost,  # substitution
                )
    return dp[m][n], dp


def backtrace_edit_operations(s1, s2, dp):
    """Reconstruct the sequence of edit operations from the DP table."""
    i, j = len(s1), len(s2)
    ops = []
    while i > 0 or j > 0:
        if i > 0 and j > 0 and s1[i - 1] == s2[j - 1] and dp[i][j] == dp[i - 1][j - 1]:
            ops.append(f"Match '{s1[i-1]}'")
            i, j = i - 1, j - 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + 1:
            ops.append(f"Substitute '{s1[i-1]}' -> '{s2[j-1]}'")
            i, j = i - 1, j - 1
        elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
            ops.append(f"Delete '{s1[i-1]}'")
            i -= 1
        else:
            ops.append(f"Insert '{s2[j-1]}'")
            j -= 1
    return list(reversed(ops))


def section10_demo():
    print("\n=== SECTION 10: Edit Distance (Levenshtein) ===")
    s1, s2 = "kitten", "sitting"
    dist, dp = edit_distance(s1, s2)
    print(f"Edit distance between '{s1}' and '{s2}': {dist}")
    print("DP table:\n", dp)
    ops = backtrace_edit_operations(s1, s2, dp)
    print("Operations:")
    for op in ops:
        print(" -", op)


# ==============================================================================
# MAIN — run all demos
# ==============================================================================

if __name__ == "__main__":
    section1_demo()
    section2_demo()
    section3_demo()
    section4_demo()
    section5_demo()
    section6_demo()
    section7_demo()
    section8_demo()
    section9_demo()
    section10_demo()



=== SECTION 1: Reading data from files ===
(Demonstration only — shows function signatures.
 In the exam, call read_txt_file('data.txt'), read_csv_file('data.csv', 'text'),
 or read_pdf_file('data.pdf') on your actual files.)
read_txt_file, read_txt_lines, read_csv_file, read_pdf_file are defined above.

=== SECTION 2: Phases of NLP ===

-- Lexical/Morphological Analysis --
Breaks text into tokens/words and studies word structure (stems, prefixes, suffixes).

-- Syntactic Analysis --
Analyzes grammatical structure of sentences (POS tags, parse trees, dependency relations).

-- Semantic Analysis --
Extracts literal/dictionary meaning of words and how they combine into sentence meaning.

-- Discourse Integration --
Interprets a sentence's meaning using the context of surrounding sentences.

-- Pragmatic Analysis --
Interprets intended meaning using real-world knowledge, tone, and context (beyond literal text).

[Lexical] Tokens: ['Apple', 'is', 'looking', 'at', 'buying', 'a', 'UK', 'sta

In [2]:
import math
import re
import numpy as np
import pandas as pd
from pypdf import PdfReader

path = "D:\\Studies\\3rd year\\Semestser 5\\Natural Language Processing\\NLP-Git\\New Text Document.txt"
path1 ="D:\\Studies\\3rd year\\Semestser 5\\Natural Language Processing\\NLP-Git\\code\\NLP_Exam_Revision_Notes.pdf" 
try:
    # with open(path, "r", encoding="utf-8") as f:
    #     # print(f.read())
    #     print([line.strip() for line in f.readlines()])
    reader = PdfReader(path1)
    text = ""
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted + "\n"
    print(text)


except Exception:
    pass

NLP Exam Revision Notes 
Byte-Pair Encoding • Edit Distance • Lesk Algorithm • TF-IDF (LSI & FSA) • N-gram Models • Regular 
Expressions • POS Tagging (HMM/CRF) • Word Sense Disambiguation  
 
1. Byte-Pair Encoding (BPE) 
1.1 What it is 
Byte-Pair Encoding is a subword tokenization algorithm. It was originally a data-compression technique (Gage, 
1994) that repeatedly replaces the most frequent pair of adjacent bytes with a new byte. Sennrich et al. (2016) 
adapted it to NLP to build a subword vocabulary for neural machine translation and other models (GPT, RoBERTa 
use byte-level BPE). 
1.2 Why subword tokenization 
● Pure word-level tokenization cannot handle out-of-vocabulary (OOV) words. 
● Pure character-level tokenization avoids OOV but produces very long sequences and loses word-level 
meaning. 
● BPE sits in between: frequent words remain whole tokens, rare/unseen words are broken into meaningful 
subword units (e.g. "unhappiness" → "un", "happi", "ness"). 
● Keeps vocabulary s

Phases in nlp are:
- Lexical/Morphological analysis
- Syntatic analysis
- Semantic analysis
- Discourse Integration
- Pragmatic analysis

In [3]:
from nltk.corpus import wordnet as wn
synsets = wn.synsets("bat")
if synsets:
    print("[Semantic] 'bank' first sense definition:", synsets[4].definition())

    # Discourse / Pragmatic are conceptual — shown via comments, not single-call code


[Semantic] 'bank' first sense definition: a club used for hitting a ball in various games


In [4]:
from nltk.corpus import wordnet as wn 
ss = wn.synsets("balls") 
if ss:
    for i in ss:
        print(i.definition())

round object that is hit or thrown or kicked in games
a solid projectile that is shot by a musket
an object with a spherical shape
the people assembled at a lavish formal dance
one of the two male reproductive glands that produce spermatozoa and secrete androgens
a spherical object used as a plaything
United States comedienne best known as the star of a popular television program (1911-1989)
a compact mass
a lavish dance requiring formal attire
a more or less rounded anatomical body or mass
the game of baseball
a pitch that is not in the strike zone
form into a ball by winding or rolling


In [5]:
#Text preprocessing

#step 1 preparing sample corpus
with open (path, "r", encoding="utf-8") as f:
    text = f.read()

#step 2 text cleaning and regular expressions
import re
lowertext = text.lower()
cleantext = re.sub(r"[^a-zA-A\s]", "", lowertext)

#step3 tokenization
from nltk.tokenize import word_tokenize 
tokens = word_tokenize(cleantext)

#step4 stopwords removal
from nltk.corpus import stopwords
sw = set(stopwords.words('english'))
swrtokens = [tok for tok in tokens if tok not in sw]

#step5 stemming
from nltk.stem import PorterStemmer
ps = PorterStemmer()
stmtokens = [ps.stem(tok) for tok in swrtokens]

#step6 lemmatization
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
lmttokens = [lemmatizer.lemmatize(tok) for tok in stmtokens]

#step7 contractions expansions
#step8 emoji corrections
#step9 spellchecker
#step10 pos tagging
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("average_perceptron_tagger")
nltk.download("average_perceptron_tagger_eng")
from nltk import pos_tag

pos_tagged = pos_tag(lmttokens)
print(pos_tagged)

[('treebank', 'NN'), ('motiv', 'NN'), ('treebank', 'NN'), ('rulebas', 'NN'), ('nlp', 'NN'), ('system', 'NN'), ('heavili', 'JJ'), ('bottleneck', 'NN'), ('linguist', 'NN'), ('ambigu', 'JJ'), ('lack', 'NN'), ('way', 'NN'), ('calcul', 'JJ'), ('probabilist', 'NN'), ('languag', 'NN'), ('pattern', 'JJ'), ('scale', 'JJ'), ('treebank', 'NN'), ('provid', 'JJ'), ('largescal', 'JJ'), ('empir', 'NN'), ('humanannot', 'NN'), ('data', 'NNS'), ('requir', 'VBP'), ('transit', 'NN'), ('nlp', 'JJ'), ('rigid', 'JJ'), ('manual', 'JJ'), ('rulewrit', 'NN'), ('statist', 'JJ'), ('machinelearningbas', 'NN'), ('par', 'NN'), ('treebank', 'VBD'), ('establish', 'VB'), ('modern', 'JJ'), ('empir', 'JJ'), ('norm', 'NN'), ('like', 'IN'), ('standard', 'JJ'), ('traintest', 'NN'), ('split', 'NN'), ('evalu', 'RB'), ('metric', 'JJ'), ('eg', 'NN'), ('parsev', 'NN'), ('model', 'NN'), ('could', 'MD'), ('univers', 'VB'), ('compar', 'JJ'), ('work', 'NN'), ('principl', 'NN'), ('behind', 'IN'), ('treebank', 'JJ'), ('construct', 'NN'

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Error loading average_perceptron_tagger: Package
[nltk_data]     'average_perceptron_tagger' not found in index
[nltk_data] Error loading average_perceptron_tagger_eng: Package
[nltk_data]     'average_perceptron_tagger_eng' not found in index


In [6]:
from sklearn.decomposition import TruncatedSVD

vocab = sorted(list(set(lmttokens)))
vocab_idx = {word:i for i,word in enumerate(vocab)}
n = len(vocab)
cmatrix = np.zeros((n,n))
for i in range(len(lmttokens)):
    neig = []
    if i > 0:
        neig.append(lmttokens[i-1])
    if i<len(lmttokens)-1:
        neig.append(lmttokens[i+1])
    for n in neig:
        cmatrix[vocab_idx[lmttokens[i]]][vocab_idx[n]] += 1

svd = TruncatedSVD(n_components=2)
val = svd.fit_transform(cmatrix)
print(val.tolist())


[[0.023932547191862295, 0.015658233451540923], [0.08113415387254529, 0.009143341203937855], [0.31746407042899366, -0.2957250174106617], [0.2929942823238556, 0.26926546495534365], [0.12636989629261525, 0.07022933622862287], [0.09803314175052016, 0.07426361152660278], [0.021022575925924502, 0.020068219374539103], [0.067466801353952, 0.020205359887652714], [0.08120535246189667, 0.08152937906062421], [0.5131949430632532, -0.5158858803091445], [0.10056637136808767, 0.06093336466866882], [0.14820011193872556, 0.09389089881330834], [0.01074399821383926, -0.005040366485252074], [0.005762698567210863, 0.002832224111256645], [0.3953974988447194, -0.36901364216833327], [0.4379087182248269, -0.43228919295849344], [0.007916616306711885, -0.00728284010504477], [0.12040770000850073, 0.09925814061038286], [0.03131512819008788, 0.02929873145776945], [0.00017485647062977127, -0.0001800340022878603], [0.0077747020111724905, -0.0083439194777687], [0.0006272965560066361, 0.0006119521664118883], [0.11453489

In [14]:
from nltk.util import ngrams
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.lm import MLE
traindata, vocab = padded_everygram_pipeline(2, [lmttokens])
model = MLE(2)
model.fit(traindata,vocab)
test_sentence = ["nlp","rigid","manual"]
test_ngrams = list(ngrams(test_sentence,2))
prob = 1
for w1,w2 in test_ngrams:
    prob *= model.score(w2,[w1])
print(f"sentence prob {prob}")
perplex = model.perplexity(test_ngrams)
print(perplex)
print(4list(traindata))

SyntaxError: invalid decimal literal (2461272401.py, line 15)

In [8]:
corpus = [
    "The king rules the kingdom with authority",
    "The queen rules the realm beside the king",
    "Data science involves statistical analysis and machine learning",
    "Machine learning algorithms process large dataset models"
]

from sklearn.feature_extraction.text import TfidfVectorizer
tfidfvectorizer = TfidfVectorizer(tokenizer=word_tokenize)
tfidfmatrix = tfidfvectorizer.fit_transform(corpus)
print(tfidfmatrix.shape)

svd = TruncatedSVD(n_components=2, random_state=42)
svdmatrix = svd.fit_transform(tfidfmatrix)
print(svdmatrix.shape)
print(tfidfvectorizer.get_feature_names_out())


(4, 22)
(4, 2)
['algorithms' 'analysis' 'and' 'authority' 'beside' 'data' 'dataset'
 'involves' 'king' 'kingdom' 'large' 'learning' 'machine' 'models'
 'process' 'queen' 'realm' 'rules' 'science' 'statistical' 'the' 'with']


C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [11]:
from nltk.corpus import treebank
nltk.download('treebank')
from nltk.tag import HiddenMarkovModelTrainer

taggedsentences = treebank.tagged_sents()
train_data = taggedsentences[:3500]
test_data = taggedsentences[3500:]
trainer = HiddenMarkovModelTrainer()
hmm_tagger = trainer.train_supervised(train_data)
print(hmm_tagger.accuracy(test_data))

test = ["the", "king","rules", "the", "kingdom"]
predicted = hmm_tagger.tag(test)
print(hmm_tagger.accuracy([predicted]))

[nltk_data] Downloading package treebank to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package treebank is already up-to-date!


0.41872398102430053
1.0


In [10]:
import sklearn_crfsuite
train_data = list(treebank.tagged_sents()[:3000])
def getfeatures(sent):
    return [{'word': w.lower(), 'is_first': i==0} for i,(w,t) in enumerate(sent)]
def getlabels(sent):
    return [t for w,t in sent]
xtrain = [getfeatures(s) for s in train_data]
ytrain = [getlabels(s) for s in train_data]
crf = sklearn_crfsuite.CRF(max_iterations = 50)
crf.fit(xtrain,ytrain)
test_tokens = ["the", "king", "rules", "the", "kingdom"]
test_features = [{'word': w.lower(), 'is_first': i == 0} for i, w in enumerate(test_tokens)]
predicted_tags = crf.predict_single(test_features)
print(list(zip(test_tokens, predicted_tags)))

[('the', 'DT'), ('king', 'NN'), ('rules', 'VBD'), ('the', 'DT'), ('kingdom', 'NN')]
